<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/03_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL READMISSION PREDICTION SYSTEM
## NOTEBOOK 03: DATA PREPROCESSING
This notebook handles data cleaning, leakage removal, feature splitting, numerical & categorical pipeline transformation, and preprocessor persistence.


### 1. IMPORT LIBRARIES


In [43]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("Libraries imported successfully.")


Libraries imported successfully.


### 2. LOAD DATASET


In [44]:
# ============================================================
# 2. LOAD DATASET
# ============================================================
df = pd.read_csv( "/content/hospital_readmission_dataset.csv")

print(f"Dataset loaded from: {data_path}")
print("Original shape:", df.shape)
display(df.head())


Dataset loaded from: /content/hospital_readmission_dataset.csv
Original shape: (8000, 17)


,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
0,P00001,2022-04-14,Spring,66,Male,South,Diabetes,5,6,Interventional,8,6,1,Medicare,Home Health,0.92,1
1,P00002,2021-09-19,Fall,55,Male,South,Diabetes,4,6,Interventional,6,4,3,Private,Home Health,0.88,1
2,P00003,2023-04-12,Spring,69,Female,West,Hypertension,6,8,Medical,9,6,2,Medicare,Skilled Nursing,0.97,1
3,P00004,2023-08-14,Summer,83,Male,South,Stroke,6,11,Medical,11,4,2,Medicare,Skilled Nursing,0.97,1
4,P00005,2021-11-05,Fall,54,Female,North,Stroke,4,10,Medical,6,2,1,Uninsured,Home Health,0.83,1


### 3. CHECK DATA


In [45]:
# ============================================================
# 3. CHECK DATA
# ============================================================

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate records:")
print(df.duplicated().sum())



Missing values:


,0
patient_id,0
admission_date,0
season,0
age,0
gender,0
region,0
primary_diagnosis,0
comorbidities_count,0
length_of_stay,0
treatment_type,0



Duplicate records:
0


### 4. REMOVE DUPLICATES


In [46]:
# ============================================================
# 4. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates()

print("\nShape after removing duplicates:", df.shape)



Shape after removing duplicates: (8000, 17)


### 5. REMOVE UNNECESSARY / RISKY FEATURES


In [47]:
# ============================================================
# 5. REMOVE UNNECESSARY / RISKY FEATURES
# ============================================================

columns_to_drop = [
    "patient_id",
    "admission_date",
    "readmission_risk_score"
]

# Drop only columns that actually exist
columns_to_drop = [
    col for col in columns_to_drop
    if col in df.columns
]

df = df.drop(columns=columns_to_drop)

print("\nRemoved columns:")
print(columns_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())



Removed columns:
['patient_id', 'admission_date', 'readmission_risk_score']

Remaining columns:
['season', 'age', 'gender', 'region', 'primary_diagnosis', 'comorbidities_count', 'length_of_stay', 'treatment_type', 'medications_count', 'followup_visits_last_year', 'prev_readmissions', 'insurance_type', 'discharge_disposition', 'label']


### 6. DEFINE TARGET


In [48]:
# ============================================================
# 6. DEFINE TARGET
# ============================================================

target = "label"

X = df.drop(columns=[target])
y = df[target]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)



Features shape: (8000, 13)
Target shape: (8000,)


### 7. IDENTIFY FEATURE TYPES


In [49]:
# ============================================================
# 7. IDENTIFY FEATURE TYPES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)



Numerical features:
['age', 'comorbidities_count', 'length_of_stay', 'medications_count', 'followup_visits_last_year', 'prev_readmissions']

Categorical features:
['season', 'gender', 'region', 'primary_diagnosis', 'treatment_type', 'insurance_type', 'discharge_disposition']


### 8. TRAIN / TEST SPLIT


In [50]:
# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)



Training data: (6400, 13)
Testing data : (1600, 13)


### 9. CHECK TARGET DISTRIBUTION


In [51]:
# ============================================================
# 9. CHECK TARGET DISTRIBUTION
# ============================================================

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).round(3))



Training target distribution:


,proportion
label,
1,0.773
0,0.227



Testing target distribution:


,proportion
label,
1,0.773
0,0.227


### 10. NUMERICAL PREPROCESSING


In [52]:
# ============================================================
# 10. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical pipeline initialized.")


Numerical pipeline initialized.


### 11. CATEGORICAL PREPROCESSING


In [53]:
# ============================================================
# 11. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

print("Categorical pipeline initialized.")


Categorical pipeline initialized.


### 12. COMBINE PREPROCESSING


In [54]:
# ============================================================
# 12. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Full preprocessor pipeline assembled.")


Full preprocessor pipeline assembled.


### 13. FIT PREPROCESSOR ONLY ON TRAINING DATA


In [55]:
# ============================================================
# 13. FIT PREPROCESSOR ONLY ON TRAINING DATA
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Preprocessor successfully fitted on training data and applied to both sets.")


Preprocessor successfully fitted on training data and applied to both sets.


### 14. CHECK PROCESSED DATA


In [56]:
# ============================================================
# 14. CHECK PROCESSED DATA
# ============================================================

print("\nProcessed training shape:")
print(X_train_processed.shape)

print("\nProcessed testing shape:")
print(X_test_processed.shape)



Processed training shape:
(6400, 40)

Processed testing shape:
(1600, 40)


### 15. SAVE PREPROCESSOR


In [57]:
# ============================================================
# 15. SAVE PREPROCESSOR
# ============================================================

save_path = "preprocessor.pkl" if not os.path.exists("/content") else "/content/preprocessor.pkl"

joblib.dump(
    preprocessor,
    save_path
)

# Also save into models folder if present
if os.path.exists("models") or os.path.exists("../models"):
    target_model_dir = "models" if os.path.exists("models") else "../models"
    joblib.dump(preprocessor, os.path.join(target_model_dir, "preprocessor.joblib"))

print(f"\nPreprocessor saved successfully to: {save_path}")



Preprocessor saved successfully to: /content/preprocessor.pkl


### 16. FINAL CHECK


In [58]:
# ============================================================
# 16. FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPROCESSING COMPLETED")
print("=" * 60)

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])
print("Processed train  :", X_train_processed.shape)
print("Processed test   :", X_test_processed.shape)

print("""
Next Step:
04_Feature_Engineering.ipynb / 05_Model_Training.ipynb
""")



DATA PREPROCESSING COMPLETED
Training samples : 6400
Testing samples  : 1600
Processed train  : (6400, 40)
Processed test   : (1600, 40)

Next Step:
04_Feature_Engineering.ipynb / 05_Model_Training.ipynb

